# 04 — Análise Gold (Alfabetização)

Este notebook realiza **somente a transformação da camada Gold** do projeto FIAP Tech Challenge Fase 2.

Fluxo executado:

1. Lê as configurações dos caminhos.
2. Carrega as tabelas da camada Silver.
3. Aplica transformações analíticas para criar métricas.
4. Gera 7 tabelas de análise em `data/gold/`.
5. Produz dicionário de dados em `docs/dicionario_dados_gold.md`.

**Tabelas Gold geradas:**
- `indicador_meta_brasil` - Indicadores nacionais de alfabetização
- `indicador_meta_uf` - Indicadores por estado
- `ranking_uf_prioritaria` - Estados com maior déficit de alfabetização
- `indicador_meta_municipio` - Indicadores por município
- `ranking_municipio_prioritario` - Municípios com maior déficit
- `evolucao_alfabetizacao` - Série histórica de evolução
- `resumo_status_meta` - Resumo de cumprimento de metas

## 1. Imports e configuração

In [ ]:
from pathlib import Path
from datetime import date, datetime

import pandas as pd

SILVER_PATH = Path("../data/silver")
GOLD_PATH = Path("../data/gold")
DOCS_PATH = Path("../docs")

EXECUTION_DATE = date.today().isoformat()

print("Silver path:", SILVER_PATH.resolve())
print("Gold path:", GOLD_PATH.resolve())
print("Docs path:", DOCS_PATH.resolve())
print("Execution date:", EXECUTION_DATE)

## 2. Funções utilitárias

In [ ]:
def localizar_parquet_mais_recente(caminho_tabela: Path) -> Path:
    """Localiza o arquivo Parquet mais recente em uma pasta."""
    arquivos = list(caminho_tabela.rglob("*.parquet"))
    
    if not arquivos:
        raise FileNotFoundError(f"Nenhum arquivo Parquet encontrado em: {caminho_tabela}")
    
    return max(arquivos, key=lambda arquivo: arquivo.stat().st_mtime)


def carregar_silver(nome_tabela: str, colunas: list[str] | None = None) -> pd.DataFrame:
    """Carrega uma tabela Silver pelo nome."""
    caminho_tabela = SILVER_PATH / nome_tabela
    arquivo = localizar_parquet_mais_recente(caminho_tabela)
    
    df = pd.read_parquet(arquivo, columns=colunas)
    
    print(f"[OK] silver.{nome_tabela} carregada")
    print(f"     Arquivo: {arquivo.name}")
    print(f"     Linhas: {len(df)} | Colunas: {len(df.columns)}")
    
    return df


def salvar_gold(df: pd.DataFrame, nome_tabela: str) -> Path:
    """Salva uma tabela Gold com particionamento por data."""
    output_dir = GOLD_PATH / nome_tabela / f"execution_date={EXECUTION_DATE}"
    output_dir.mkdir(parents=True, exist_ok=True)
    
    output_file = output_dir / f"{nome_tabela}.parquet"
    df.to_parquet(output_file, index=False)
    
    print(f"[OK] gold.{nome_tabela} salva em: {output_file}")
    print(f"     Linhas: {len(df)} | Colunas: {len(df.columns)}")
    
    return output_file


def aplicar_status_meta(df: pd.DataFrame) -> pd.DataFrame:
    """Calcula status de cumprimento de meta."""
    df = df.copy()
    
    df["distancia_meta"] = df["taxa_alfabetizacao"] - df["meta_alfabetizacao"]
    df["flag_meta_atingida"] = df["distancia_meta"] >= 0
    
    df.loc[
        df["taxa_alfabetizacao"].isna() | df["meta_alfabetizacao"].isna(),
        "flag_meta_atingida"
    ] = pd.NA
    
    df["status_meta"] = "Sem informação"
    df.loc[df["flag_meta_atingida"] == True, "status_meta"] = "Meta atingida"
    df.loc[df["flag_meta_atingida"] == False, "status_meta"] = "Abaixo da meta"
    
    return df

## 3. Carregar tabelas Silver

In [ ]:
print("Carregando tabelas Silver necessárias para Gold...")
print("\n" + "=" * 80)

df_fato_resultado_brasil = carregar_silver("fato_resultado_brasil")
print()

df_fato_resultado_uf = carregar_silver("fato_resultado_uf")
print()

df_fato_resultado_municipio = carregar_silver("fato_resultado_municipio")
print()

df_fato_meta_anual_brasil = carregar_silver("fato_meta_anual_brasil")
print()

df_fato_meta_anual_uf = carregar_silver("fato_meta_anual_uf")
print()

df_fato_meta_anual_municipio = carregar_silver("fato_meta_anual_municipio")
print()

print("=" * 80)
print("Todas as tabelas Silver carregadas com sucesso!")

## 4. Indicador Meta Brasil

In [ ]:
# Merge entre resultado e meta para Brasil
df_indicador_meta_brasil = (
    df_fato_resultado_brasil
    .merge(
        df_fato_meta_anual_brasil,
        on=["ano", "rede"],
        how="inner",
        suffixes=("_resultado", "_meta")
    )
)

# Aplicar status de meta
df_indicador_meta_brasil = aplicar_status_meta(df_indicador_meta_brasil)

# Ordenar
df_indicador_meta_brasil = (
    df_indicador_meta_brasil
    .sort_values(["ano", "rede"])
    .reset_index(drop=True)
)

# Salvar
salvar_gold(df_indicador_meta_brasil, "indicador_meta_brasil")

print("\nPrimeiras linhas de indicador_meta_brasil:")
display(df_indicador_meta_brasil.head(10))

## 5. Indicador Meta UF

In [ ]:
# Merge entre resultado e meta para UF
df_indicador_meta_uf = (
    df_fato_resultado_uf
    .merge(
        df_fato_meta_anual_uf,
        on=["ano", "sigla_uf", "rede"],
        how="inner",
        suffixes=("_resultado", "_meta")
    )
)

# Aplicar status de meta
df_indicador_meta_uf = aplicar_status_meta(df_indicador_meta_uf)

# Ordenar
df_indicador_meta_uf = (
    df_indicador_meta_uf
    .sort_values(["ano", "sigla_uf", "rede"])
    .reset_index(drop=True)
)

# Salvar
salvar_gold(df_indicador_meta_uf, "indicador_meta_uf")

print("\nPrimeiras linhas de indicador_meta_uf:")
display(df_indicador_meta_uf.head(10))

## 6. Ranking UF Prioritária

In [ ]:
# Ranking de UFs com maior déficit (distância negativa da meta)
df_ranking_uf_prioritaria = (
    df_indicador_meta_uf[df_indicador_meta_uf["status_meta"] == "Abaixo da meta"]
    .groupby(["ano", "sigla_uf"])
    .agg({
        "distancia_meta": "mean",
        "taxa_alfabetizacao": "mean",
    })
    .reset_index()
    .sort_values(["ano", "distancia_meta"])
    .reset_index(drop=True)
)

# Adicionar ranking por ano
df_ranking_uf_prioritaria["ranking"] = (
    df_ranking_uf_prioritaria
    .groupby("ano")
    .cumcount() + 1
)

# Limitar aos 10 primeiros
df_ranking_uf_prioritaria = df_ranking_uf_prioritaria[
    df_ranking_uf_prioritaria["ranking"] <= 10
]

# Salvar
salvar_gold(df_ranking_uf_prioritaria, "ranking_uf_prioritaria")

print("\nRanking de UFs prioritárias (maiores déficits):")
display(df_ranking_uf_prioritaria.head(15))

## 7. Indicador Meta Município

In [ ]:
# Merge entre resultado e meta para Município
df_indicador_meta_municipio = (
    df_fato_resultado_municipio
    .merge(
        df_fato_meta_anual_municipio,
        on=["ano", "id_municipio", "rede"],
        how="inner",
        suffixes=("_resultado", "_meta")
    )
)

# Aplicar status de meta
df_indicador_meta_municipio = aplicar_status_meta(df_indicador_meta_municipio)

# Ordenar
df_indicador_meta_municipio = (
    df_indicador_meta_municipio
    .sort_values(["ano", "id_municipio", "rede"])
    .reset_index(drop=True)
)

# Salvar
salvar_gold(df_indicador_meta_municipio, "indicador_meta_municipio")

print(f"\nIndicador Meta Município: {len(df_indicador_meta_municipio)} registros")
print("\nPrimeiras linhas:")
display(df_indicador_meta_municipio.head(10))

## 8. Ranking Município Prioritário

In [ ]:
# Ranking de municípios com maior déficit
df_ranking_municipio_prioritario = (
    df_indicador_meta_municipio[
        df_indicador_meta_municipio["status_meta"] == "Abaixo da meta"
    ]
    .groupby(["ano", "id_municipio"])
    .agg({
        "distancia_meta": "mean",
        "taxa_alfabetizacao": "mean",
    })
    .reset_index()
    .sort_values(["ano", "distancia_meta"])
    .reset_index(drop=True)
)

# Adicionar ranking por ano
df_ranking_municipio_prioritario["ranking"] = (
    df_ranking_municipio_prioritario
    .groupby("ano")
    .cumcount() + 1
)

# Limitar aos 20 primeiros
df_ranking_municipio_prioritario = df_ranking_municipio_prioritario[
    df_ranking_municipio_prioritario["ranking"] <= 20
]

# Salvar
salvar_gold(df_ranking_municipio_prioritario, "ranking_municipio_prioritario")

print("\nRanking de municípios prioritários (maiores déficits):")
display(df_ranking_municipio_prioritario.head(20))

## 9. Evolução da Alfabetização

In [ ]:
# Série histórica de evolução de alfabetização
df_evolucao_alfabetizacao = (
    df_fato_resultado_brasil
    .groupby("ano")
    .agg({
        "taxa_alfabetizacao": "mean",
        "media_portugues": "mean",
    })
    .reset_index()
    .sort_values("ano")
)

# Calcular variação
df_evolucao_alfabetizacao["variacao_taxa"] = (
    df_evolucao_alfabetizacao["taxa_alfabetizacao"]
    .diff()
)

df_evolucao_alfabetizacao["variacao_media_portugues"] = (
    df_evolucao_alfabetizacao["media_portugues"]
    .diff()
)

# Salvar
salvar_gold(df_evolucao_alfabetizacao, "evolucao_alfabetizacao")

print("\nSérie histórica de evolução:")
display(df_evolucao_alfabetizacao)

## 10. Resumo Status Meta

In [ ]:
# Resumo de cumprimento de metas por nível de agregação
df_resumo_status_meta_brasil = (
    df_indicador_meta_brasil
    .groupby(["ano", "status_meta"])
    .size()
    .reset_index(name="quantidade")
)
df_resumo_status_meta_brasil["nivel"] = "Brasil"

df_resumo_status_meta_uf = (
    df_indicador_meta_uf
    .groupby(["ano", "status_meta"])
    .size()
    .reset_index(name="quantidade")
)
df_resumo_status_meta_uf["nivel"] = "UF"

df_resumo_status_meta_municipio = (
    df_indicador_meta_municipio
    .groupby(["ano", "status_meta"])
    .size()
    .reset_index(name="quantidade")
)
df_resumo_status_meta_municipio["nivel"] = "Município"

# Consolidar
df_resumo_status_meta = pd.concat(
    [
        df_resumo_status_meta_brasil,
        df_resumo_status_meta_uf,
        df_resumo_status_meta_municipio,
    ],
    ignore_index=True
)

df_resumo_status_meta = (
    df_resumo_status_meta
    .sort_values(["ano", "nivel", "status_meta"])
    .reset_index(drop=True)
)

# Salvar
salvar_gold(df_resumo_status_meta, "resumo_status_meta")

print("\nResumo de cumprimento de metas:")
display(df_resumo_status_meta)

## 11. Resumo Final da Análise Gold

In [ ]:
print("=" * 80)
print("RESUMO DA ANÁLISE GOLD")
print("=" * 80)
print(f"\nData de execução: {EXECUTION_DATE}")
print(f"\nTabelas Gold geradas:")
print("  1. indicador_meta_brasil")
print("  2. indicador_meta_uf")
print("  3. ranking_uf_prioritaria")
print("  4. indicador_meta_municipio")
print("  5. ranking_municipio_prioritario")
print("  6. evolucao_alfabetizacao")
print("  7. resumo_status_meta")

print(f"\nArquivos salvos em: {GOLD_PATH.resolve()}")
print("\nStatus: Análise Gold concluída com sucesso!")